# Question 4 — Evaluating an LLM sentiment classifier
Zaid Bin Naveed · 23K-0028

Every metric is computed twice: once by hand from its formula, once with a standard library (scikit-learn, NLTK, rouge-score). An `assert` confirms they agree.

In [1]:
import json
import math
from pathlib import Path

import nltk
import numpy as np
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

for pkg in ("wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)

CLASSES = ["Positive", "Negative", "Neutral"]
STAGES = {
    "stage1_zero_shot": np.array([[100, 5, 15], [8, 80, 12], [20, 10, 50]]),
    "stage2_few_shot": np.array([[106, 4, 10], [6, 84, 10], [9, 7, 64]]),
}
results = {}

## Q1–Q3, Q5: accuracy with 95% CI, per-class P/R/F1, macro / weighted / micro F1

* Accuracy $= \frac{\sum_i C_{ii}}{n}$.
* **95% Wald interval:** $\hat p \pm z_{0.975}\sqrt{\hat p(1-\hat p)/n}$ with $z_{0.975}=1.96$. A Wilson interval is printed too as a sanity check, because Wald can misbehave near 0 or 1 (not an issue at n = 300, p ≈ 0.8).
* Precision$_c = C_{cc}/\text{column}_c$, Recall$_c = C_{cc}/\text{row}_c$, F1$_c = 2PR/(P+R)$.
* Macro F1 = unweighted mean of F1$_c$; Weighted F1 = $\sum_c (\text{support}_c/n)\,F1_c$; Micro F1 pools TP/FP/FN over classes, so for single-label multi-class it equals accuracy.

In [2]:
def expand(cm):
    y_true, y_pred = [], []
    for i in range(3):
        for j in range(3):
            y_true += [i] * cm[i, j]
            y_pred += [j] * cm[i, j]
    return np.array(y_true), np.array(y_pred)


def wald(p, n, z=1.96):
    half = z * math.sqrt(p * (1 - p) / n)
    return p - half, p + half, half


def wilson(p, n, z=1.96):
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


def evaluate(name, cm):
    n = cm.sum()
    tp, row, col = np.diag(cm), cm.sum(1), cm.sum(0)
    acc = tp.sum() / n
    lo, hi, half = wald(acc, n)
    prec, rec = tp / col, tp / row
    f1 = 2 * prec * rec / (prec + rec)
    macro, weighted = f1.mean(), (row / n * f1).sum()
    micro_p = micro_r = tp.sum() / n  # sum(FP) = sum(FN) = n - sum(TP)
    micro = 2 * micro_p * micro_r / (micro_p + micro_r)

    y_true, y_pred = expand(cm)
    lib_p, lib_r, lib_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], zero_division=0)
    checks = {
        "accuracy": (acc, accuracy_score(y_true, y_pred)),
        "macro_f1": (macro, f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": (weighted, f1_score(y_true, y_pred, average="weighted")),
        "micro_f1": (micro, f1_score(y_true, y_pred, average="micro")),
    }
    assert np.allclose(prec, lib_p) and np.allclose(rec, lib_r) and np.allclose(f1, lib_f1)
    assert all(abs(a - b) < 1e-12 for a, b in checks.values())

    print(f"=== {name} (n = {n}) ===")
    print(f"accuracy = {tp.sum()}/{n} = {acc:.4f}")
    print(f"Wald 95% CI = {acc:.4f} ± 1.96*sqrt({acc:.4f}*{1 - acc:.4f}/{n}) = {acc:.4f} ± {half:.4f} = [{lo:.4f}, {hi:.4f}]")
    print("Wilson 95% CI (check) = [{:.4f}, {:.4f}]".format(*wilson(acc, n)))
    for c, cls in enumerate(CLASSES):
        print(f"{cls:9s} P = {tp[c]}/{col[c]} = {prec[c]:.4f}  R = {tp[c]}/{row[c]} = {rec[c]:.4f}  F1 = {f1[c]:.4f}")
    for k, (manual, lib) in checks.items():
        print(f"{k:12s} manual = {manual:.4f}   sklearn = {lib:.4f}")
    return {
        "n": int(n), "correct": int(tp.sum()), "accuracy": acc, "wald_ci": [lo, hi], "wald_half_width": half,
        "wilson_ci": list(wilson(acc, n)), "tp": tp.tolist(), "row_totals": row.tolist(), "col_totals": col.tolist(),
        "precision": prec.tolist(), "recall": rec.tolist(), "f1": f1.tolist(),
        "macro_f1": macro, "weighted_f1": weighted, "micro_f1": micro, "confusion": cm.tolist(),
    }


for name, cm in STAGES.items():
    results[name] = evaluate(name, cm)
    print()

=== stage1_zero_shot (n = 300) ===
accuracy = 230/300 = 0.7667
Wald 95% CI = 0.7667 ± 1.96*sqrt(0.7667*0.2333/300) = 0.7667 ± 0.0479 = [0.7188, 0.8145]
Wilson 95% CI (check) = [0.7156, 0.8110]
Positive  P = 100/128 = 0.7812  R = 100/120 = 0.8333  F1 = 0.8065
Negative  P = 80/95 = 0.8421  R = 80/100 = 0.8000  F1 = 0.8205
Neutral   P = 50/77 = 0.6494  R = 50/80 = 0.6250  F1 = 0.6369
accuracy     manual = 0.7667   sklearn = 0.7667
macro_f1     manual = 0.7546   sklearn = 0.7546
weighted_f1  manual = 0.7659   sklearn = 0.7659
micro_f1     manual = 0.7667   sklearn = 0.7667

=== stage2_few_shot (n = 300) ===
accuracy = 254/300 = 0.8467
Wald 95% CI = 0.8467 ± 1.96*sqrt(0.8467*0.1533/300) = 0.8467 ± 0.0408 = [0.8059, 0.8874]
Wilson 95% CI (check) = [0.8015, 0.8830]
Positive  P = 106/121 = 0.8760  R = 106/120 = 0.8833  F1 = 0.8797
Negative  P = 84/95 = 0.8842  R = 84/100 = 0.8400  F1 = 0.8615
Neutral   P = 64/84 = 0.7619  R = 64/80 = 0.8000  F1 = 0.7805
accuracy     manual = 0.8467   sklearn =

### Stage 1 → Stage 2 comparison and the error pattern

In [3]:
s1, s2 = results["stage1_zero_shot"], results["stage2_few_shot"]
for k in ("accuracy", "macro_f1", "weighted_f1", "micro_f1"):
    print(f"{k:12s} {s1[k]:.4f} -> {s2[k]:.4f}  ({s2[k] - s1[k]:+.4f})")
for c, cls in enumerate(CLASSES):
    print(f"F1 {cls:9s} {s1['f1'][c]:.4f} -> {s2['f1'][c]:.4f}  ({s2['f1'][c] - s1['f1'][c]:+.4f})")

cm1 = STAGES["stage1_zero_shot"]
off = {f"{CLASSES[i]}->{CLASSES[j]}": int(cm1[i, j]) for i in range(3) for j in range(3) if i != j}
print("\nStage 1 errors (actual->predicted), largest first:", sorted(off.items(), key=lambda kv: -kv[1]))
neutral_involved = sum(v for k, v in off.items() if "Neutral" in k)
print(f"errors involving Neutral: {neutral_involved}/{sum(off.values())}")
overlap = s1["wald_ci"][1] >= s2["wald_ci"][0]
print(f"Stage 1 CI {np.round(s1['wald_ci'], 4)} vs Stage 2 CI {np.round(s2['wald_ci'], 4)} overlap: {overlap}")

# Paired comparison: the same 300 reviews were scored twice, but per-review outcomes are not given,
# so McNemar cannot be computed exactly. The overlap check above is the conservative alternative.
results["comparison"] = {"stage1_errors": off, "errors_involving_neutral": neutral_involved,
                         "total_errors_stage1": sum(off.values()), "ci_overlap": bool(overlap)}

accuracy     0.7667 -> 0.8467  (+0.0800)
macro_f1     0.7546 -> 0.8406  (+0.0859)
weighted_f1  0.7659 -> 0.8472  (+0.0812)
micro_f1     0.7667 -> 0.8467  (+0.0800)
F1 Positive  0.8065 -> 0.8797  (+0.0732)
F1 Negative  0.8205 -> 0.8615  (+0.0410)
F1 Neutral   0.6369 -> 0.7805  (+0.1435)

Stage 1 errors (actual->predicted), largest first: [('Neutral->Positive', 20), ('Positive->Neutral', 15), ('Negative->Neutral', 12), ('Neutral->Negative', 10), ('Negative->Positive', 8), ('Positive->Negative', 5)]
errors involving Neutral: 57/70
Stage 1 CI [0.7188 0.8145] vs Stage 2 CI [0.8059 0.8874] overlap: True


## Q6: prompt stability over 5 orderings
Sample standard deviation $s=\sqrt{\frac{1}{n-1}\sum (x_i-\bar x)^2}$ (Bessel-corrected) and population $\sigma=\sqrt{\frac{1}{n}\sum (x_i-\bar x)^2}$.
**I report the sample std**: the five orderings are a sample from the many possible orderings, and we want to estimate the spread over all of them.

In [4]:
runs = np.array([0.847, 0.840, 0.853, 0.837, 0.850])
mean = runs.sum() / len(runs)
dev2 = (runs - mean) ** 2
s_sample = math.sqrt(dev2.sum() / (len(runs) - 1))
s_pop = math.sqrt(dev2.sum() / len(runs))
assert abs(s_sample - runs.std(ddof=1)) < 1e-15 and abs(s_pop - runs.std(ddof=0)) < 1e-15

print(f"mean = {runs.sum():.3f}/5 = {mean:.4f}")
print("deviations   =", np.round(runs - mean, 4))
print("squared devs =", np.round(dev2, 8), " sum =", round(dev2.sum(), 8))
print(f"sample std (n-1)     = sqrt({dev2.sum():.8f}/4) = {s_sample:.4f}")
print(f"population std (n)   = sqrt({dev2.sum():.8f}/5) = {s_pop:.4f}")
print(f"report: {mean:.4f} ± {s_sample:.4f}   (range {runs.min():.3f}–{runs.max():.3f}, spread {runs.max() - runs.min():.3f})")
print(f"coefficient of variation = {s_sample / mean * 100:.2f}%")
print(f"single-run Wald half-width at n=300 ≈ {wald(mean, 300)[2]:.4f}")
results["stability"] = {"runs": runs.tolist(), "mean": mean, "sum_sq_dev": dev2.sum(), "std_sample": s_sample,
                        "std_population": s_pop, "cv_percent": s_sample / mean * 100, "range": [runs.min(), runs.max()],
                        "single_run_wald_half_width": wald(mean, 300)[2]}

mean = 4.227/5 = 0.8454
deviations   = [ 0.0016 -0.0054  0.0076 -0.0084  0.0046]
squared devs = [2.560e-06 2.916e-05 5.776e-05 7.056e-05 2.116e-05]  sum = 0.0001812
sample std (n-1)     = sqrt(0.00018120/4) = 0.0067
population std (n)   = sqrt(0.00018120/5) = 0.0060
report: 0.8454 ± 0.0067   (range 0.837–0.853, spread 0.016)
coefficient of variation = 0.80%
single-run Wald half-width at n=300 ≈ 0.0409


## Q7: BLEU-4, ROUGE-1/2/L, METEOR

* **BLEU-4** $= BP \cdot \exp\big(\sum_{n=1}^{4}\tfrac14 \ln p_n\big)$ with clipped n-gram precisions $p_n$ and $BP = e^{1 - r/c}$ when $c < r$.
  *Smoothing choice:* NLTK `SmoothingFunction().method1` (adds ε to zero counts). Every $p_n$ here is non-zero, so smoothed and unsmoothed BLEU are identical; both are shown to prove it.
* **ROUGE-N** recall/precision/F over n-gram overlaps; **ROUGE-L** uses the longest common subsequence (LCS). Library: Google's `rouge-score` (no stemming, so 'late' ≠ 'slow').
* **METEOR** (NLTK, α = 0.9, β = 3, γ = 0.5): $F_{mean} = \frac{PR}{\alpha P + (1-\alpha)R}$, penalty $=\gamma(\text{chunks}/m)^\beta$, score $= F_{mean}(1-\text{penalty})$. NLTK matches exact words, then stems, then WordNet synonyms.

In [5]:
reference = "the delivery was late but the product quality is good".split()
candidate = "the product quality is good but delivery was slow".split()
c_len, r_len = len(candidate), len(reference)


def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def clipped_precision(n):
    cand, ref = ngrams(candidate, n), ngrams(reference, n)
    matched = [g for g in set(cand) if g in ref]
    hits = sum(min(cand.count(g), ref.count(g)) for g in set(cand))
    return hits, len(cand), sorted(" ".join(g) for g in matched)


bleu_parts = {}
for n in range(1, 5):
    hits, total, matched = clipped_precision(n)
    bleu_parts[n] = (hits, total)
    print(f"p_{n} = {hits}/{total} = {hits / total:.4f}   matched: {matched}")

bp = 1.0 if c_len > r_len else math.exp(1 - r_len / c_len)
log_mean = sum(0.25 * math.log(h / t) for h, t in bleu_parts.values())
bleu_manual = bp * math.exp(log_mean)
bleu_nltk = sentence_bleu([reference], candidate, weights=(0.25,) * 4, smoothing_function=SmoothingFunction().method1)
bleu_nltk_raw = sentence_bleu([reference], candidate, weights=(0.25,) * 4)
print(f"c = {c_len}, r = {r_len}, BP = exp(1 - {r_len}/{c_len}) = {bp:.4f}")
print(f"geo-mean precision = exp({log_mean:.4f}) = {math.exp(log_mean):.4f}")
print(f"BLEU-4 manual = {bleu_manual:.4f} | NLTK method1 = {bleu_nltk:.4f} | NLTK unsmoothed = {bleu_nltk_raw:.4f}")
assert abs(bleu_manual - bleu_nltk) < 1e-9 and abs(bleu_manual - bleu_nltk_raw) < 1e-9

p_1 = 8/9 = 0.8889   matched: ['but', 'delivery', 'good', 'is', 'product', 'quality', 'the', 'was']
p_2 = 5/8 = 0.6250   matched: ['delivery was', 'is good', 'product quality', 'quality is', 'the product']
p_3 = 3/7 = 0.4286   matched: ['product quality is', 'quality is good', 'the product quality']
p_4 = 2/6 = 0.3333   matched: ['product quality is good', 'the product quality is']
c = 9, r = 10, BP = exp(1 - 10/9) = 0.8948
geo-mean precision = exp(-0.6334) = 0.5308
BLEU-4 manual = 0.4750 | NLTK method1 = 0.4750 | NLTK unsmoothed = 0.4750


In [6]:
def prf(overlap, cand_total, ref_total):
    p, r = overlap / cand_total, overlap / ref_total
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def lcs(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            dp[i + 1][j + 1] = dp[i][j] + 1 if x == y else max(dp[i][j + 1], dp[i + 1][j])
    return dp[-1][-1]


rouge_manual = {
    "rouge1": prf(bleu_parts[1][0], c_len, r_len),
    "rouge2": prf(bleu_parts[2][0], c_len - 1, r_len - 1),
    "rougeL": prf(lcs(candidate, reference), c_len, r_len),
}
lib = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False).score(" ".join(reference), " ".join(candidate))
print(f"LCS length = {lcs(candidate, reference)}  (e.g. 'the product quality is good')")
for k, (p, r, f) in rouge_manual.items():
    s = lib[k]
    print(f"{k}: manual P={p:.4f} R={r:.4f} F={f:.4f} | rouge-score P={s.precision:.4f} R={s.recall:.4f} F={s.fmeasure:.4f}")
    assert np.allclose([p, r, f], [s.precision, s.recall, s.fmeasure])

LCS length = 5  (e.g. 'the product quality is good')
rouge1: manual P=0.8889 R=0.8000 F=0.8421 | rouge-score P=0.8889 R=0.8000 F=0.8421
rouge2: manual P=0.6250 R=0.5556 F=0.5882 | rouge-score P=0.6250 R=0.5556 F=0.5882
rougeL: manual P=0.5556 R=0.5000 F=0.5263 | rouge-score P=0.5556 R=0.5000 F=0.5263


In [7]:
from nltk.corpus import wordnet
from nltk.translate.meteor_score import align_words

alpha_m, beta_m, gamma_m = 0.9, 3.0, 0.5
matches, _, _ = align_words(candidate, reference)  # (candidate index, reference index) pairs
matches = sorted(matches)
m = len(matches)


def count_chunks(pairs):
    chunks, i = 0, 0
    while i < len(pairs):
        chunks += 1
        while i + 1 < len(pairs) and pairs[i + 1][0] == pairs[i][0] + 1 and pairs[i + 1][1] == pairs[i][1] + 1:
            i += 1
        i += 1
    return chunks


chunks = count_chunks(matches)
P, Rm = m / c_len, m / r_len
fmean = P * Rm / (alpha_m * P + (1 - alpha_m) * Rm)
penalty = gamma_m * (chunks / m) ** beta_m
meteor_manual = fmean * (1 - penalty)
meteor_nltk = meteor_score([reference], candidate)

print("alignment (candidate word -> reference position):", [(candidate[i], j) for i, j in matches])
print("'slow' and 'late' share a WordNet synset:",
      bool(set(wordnet.synsets("slow")) & set(wordnet.synsets("late"))))
print(f"m = {m}, P = {m}/{c_len} = {P:.4f}, R = {m}/{r_len} = {Rm:.4f}")
print(f"Fmean = P*R/(0.9P + 0.1R) = {fmean:.4f}")
print(f"chunks = {chunks}, penalty = 0.5*({chunks}/{m})^3 = {penalty:.4f}")
print(f"METEOR manual = {meteor_manual:.4f} | NLTK = {meteor_nltk:.4f}")
assert abs(meteor_manual - meteor_nltk) < 1e-9

results["text_metrics"] = {
    "reference": " ".join(reference), "candidate": " ".join(candidate), "c_len": c_len, "r_len": r_len,
    "bleu_ngram": {n: list(v) for n, v in bleu_parts.items()}, "brevity_penalty": bp, "bleu4": bleu_manual,
    "bleu_smoothing": "NLTK SmoothingFunction().method1 (identical to unsmoothed here: no zero n-gram counts)",
    "rouge": {k: dict(zip(("precision", "recall", "f1"), v)) for k, v in rouge_manual.items()},
    "lcs": lcs(candidate, reference),
    "meteor": {"matches": m, "alignment": [[candidate[i], j] for i, j in matches], "chunks": chunks,
               "precision": P, "recall": Rm, "fmean": fmean, "penalty": penalty, "score": meteor_manual},
}

alignment (candidate word -> reference position): [('the', 5), ('product', 6), ('quality', 7), ('is', 8), ('good', 9), ('but', 4), ('delivery', 1), ('was', 2)]
'slow' and 'late' share a WordNet synset: False
m = 8, P = 8/9 = 0.8889, R = 8/10 = 0.8000
Fmean = P*R/(0.9P + 0.1R) = 0.8081
chunks = 3, penalty = 0.5*(3/8)^3 = 0.0264
METEOR manual = 0.7868 | NLTK = 0.7868


## Q8: perplexity of the generated justification
$\text{PPL} = \exp\!\big(-\tfrac{1}{N}\sum_{i=1}^{N}\ln p_i\big)$ — equivalently the inverse geometric mean of the token probabilities.

In [8]:
probs = np.array([0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20])
logs = np.log(probs)
nll = -logs.sum() / len(probs)
ppl = math.exp(nll)
geo = np.prod(probs) ** (1 / len(probs))
assert abs(ppl - 1 / geo) < 1e-12

for p, lp in zip(probs, logs):
    print(f"ln({p:.2f}) = {lp:+.4f}")
print(f"sum ln p = {logs.sum():.4f}; average NLL = {-logs.sum():.4f}/9 = {nll:.4f} nats")
print(f"PPL = exp({nll:.4f}) = {ppl:.4f};  check 1/geo-mean = 1/{geo:.4f} = {1 / geo:.4f}")
print(f"lowest-confidence tokens: {[(i + 1, float(p)) for i, p in enumerate(probs) if p <= 0.30]}")
results["perplexity"] = {"probs": probs.tolist(), "ln_probs": logs.tolist(), "sum_ln": logs.sum(),
                         "avg_nll": nll, "ppl": ppl, "geo_mean": geo}

ln(0.60) = -0.5108
ln(0.40) = -0.9163
ln(0.50) = -0.6931
ln(0.80) = -0.2231
ln(0.70) = -0.3567
ln(0.30) = -1.2040
ln(0.25) = -1.3863
ln(0.50) = -0.6931
ln(0.20) = -1.6094
sum ln p = -7.5929; average NLL = 7.5929/9 = 0.8437 nats
PPL = exp(0.8437) = 2.3249;  check 1/geo-mean = 1/0.4301 = 2.3249
lowest-confidence tokens: [(6, 0.3), (7, 0.25), (9, 0.2)]


In [9]:
Path("results").mkdir(exist_ok=True)
Path("results/q4_results.json").write_text(json.dumps(results, indent=2, default=float))
print("saved results/q4_results.json")

saved results/q4_results.json
